## OLALEYE AYOMIDE SAMUEL
## MACHINE LEANRING/ARTIFICIAL INTELLIGENCE
## PROFESSIONAL ML WORKFLOW ASSIGNMENT

In [ ]:
# ============================================================
# SUPERSTORE MACHINE LEARNING PROJECT
# Predicting Profit using Random Forest and Gradient Boosting
# ============================================================

# Import Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv("SampleSuperstore.csv")

print("Dataset Shape:", df.shape)
print(df.head())

# ============================================================
# FEATURE ENGINEERING
# ============================================================

# Check missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Remove duplicates
df = df.drop_duplicates()

# Target Variable
y = df["Profit"]

# Features
X = df.drop("Profit", axis=1)

# Categorical and Numerical Columns
categorical_cols = X.select_dtypes(include=["object"]).columns
numerical_cols = X.select_dtypes(exclude=["object"]).columns

# One-Hot Encoding
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        )
    ],
    remainder="passthrough"
)

# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# ============================================================
# RANDOM FOREST BASELINE MODEL
# ============================================================

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        random_state=42
    ))
])

rf_pipeline.fit(X_train, y_train)

rf_pred = rf_pipeline.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print("\n==============================")
print("BASELINE RANDOM FOREST")
print("==============================")
print("MAE :", rf_mae)
print("RMSE:", rf_rmse)
print("R²  :", rf_r2)

# ============================================================
# TUNED RANDOM FOREST
# ============================================================

rf_param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [10, 20, None],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

rf_grid = GridSearchCV(
    rf_pipeline,
    rf_param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

best_rf = rf_grid.best_estimator_

rf_tuned_pred = best_rf.predict(X_test)

rf_tuned_mae = mean_absolute_error(y_test, rf_tuned_pred)
rf_tuned_rmse = np.sqrt(mean_squared_error(y_test, rf_tuned_pred))
rf_tuned_r2 = r2_score(y_test, rf_tuned_pred)

print("\n==============================")
print("TUNED RANDOM FOREST")
print("==============================")
print("Best Parameters:")
print(rf_grid.best_params_)

print("MAE :", rf_tuned_mae)
print("RMSE:", rf_tuned_rmse)
print("R²  :", rf_tuned_r2)

# ============================================================
# BASELINE GRADIENT BOOSTING
# ============================================================

gb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(
        random_state=42
    ))
])

gb_pipeline.fit(X_train, y_train)

gb_pred = gb_pipeline.predict(X_test)

gb_mae = mean_absolute_error(y_test, gb_pred)
gb_rmse = np.sqrt(mean_squared_error(y_test, gb_pred))
gb_r2 = r2_score(y_test, gb_pred)

print("\n==============================")
print("BASELINE GRADIENT BOOSTING")
print("==============================")
print("MAE :", gb_mae)
print("RMSE:", gb_rmse)
print("R²  :", gb_r2)

# ============================================================
# TUNED GRADIENT BOOSTING
# ============================================================

gb_param_grid = {
    "model__n_estimators": [100, 200],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_depth": [3, 5]
}

gb_grid = GridSearchCV(
    gb_pipeline,
    gb_param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

gb_grid.fit(X_train, y_train)

best_gb = gb_grid.best_estimator_

gb_tuned_pred = best_gb.predict(X_test)

gb_tuned_mae = mean_absolute_error(y_test, gb_tuned_pred)
gb_tuned_rmse = np.sqrt(mean_squared_error(y_test, gb_tuned_pred))
gb_tuned_r2 = r2_score(y_test, gb_tuned_pred)

print("\n==============================")
print("TUNED GRADIENT BOOSTING")
print("==============================")
print("Best Parameters:")
print(gb_grid.best_params_)

print("MAE :", gb_tuned_mae)
print("RMSE:", gb_tuned_rmse)
print("R²  :", gb_tuned_r2)

# ============================================================
# MODEL COMPARISON
# ============================================================

results = pd.DataFrame({
    "Model": [
        "Random Forest Baseline",
        "Random Forest Tuned",
        "Gradient Boosting Baseline",
        "Gradient Boosting Tuned"
    ],
    "MAE": [
        rf_mae,
        rf_tuned_mae,
        gb_mae,
        gb_tuned_mae
    ],
    "RMSE": [
        rf_rmse,
        rf_tuned_rmse,
        gb_rmse,
        gb_tuned_rmse
    ],
    "R2 Score": [
        rf_r2,
        rf_tuned_r2,
        gb_r2,
        gb_tuned_r2
    ]
})

print("\n==============================")
print("MODEL COMPARISON")
print("==============================")
print(results)

# ============================================================
# BEST MODEL
# ============================================================

best_model = results.loc[
    results["R2 Score"].idxmax()
]

print("\nBest Performing Model:")
print(best_model)

Dataset Shape: (9994, 13)
        Ship Mode    Segment        Country             City       State  \
0    Second Class   Consumer  United States        Henderson    Kentucky   
1    Second Class   Consumer  United States        Henderson    Kentucky   
2    Second Class  Corporate  United States      Los Angeles  California   
3  Standard Class   Consumer  United States  Fort Lauderdale     Florida   
4  Standard Class   Consumer  United States  Fort Lauderdale     Florida   

   Postal Code Region         Category Sub-Category     Sales  Quantity  \
0        42420  South        Furniture    Bookcases  261.9600         2   
1        42420  South        Furniture       Chairs  731.9400         3   
2        90036   West  Office Supplies       Labels   14.6200         2   
3        33311  South        Furniture       Tables  957.5775         5   
4        33311  South  Office Supplies      Storage   22.3680         2   

   Discount    Profit  
0      0.00   41.9136  
1      0.00  219.5